# Tema 4. Bagging y random forest

**Técnicas Avanzadas de Predicción en Negocios · Cuaderno de práctica (nivel medio)**

Datos: Encuesta Cuatrienal de Estructura Salarial 2022 (INE), microdatos.

## Qué vamos a hacer

Este cuaderno sigue a `tema_04_arboles`, que comparaba árboles de distinto tamaño con el conjunto de test. Aquí elegimos el tamaño del árbol con validación cruzada en el entrenamiento, promediamos muchos árboles y comparamos los ensambles con la regresión lineal.

0. **Preparación.** Datos, predictores, preprocesador, particiones de validación cruzada, modelo nulo y modelo base (regresión lineal).
1. **Del árbol al random forest.** Elegimos el tamaño del árbol con `GridSearchCV`, promediamos 100 árboles (bagging) y sorteamos las columnas de cada corte (random forest).
2. **Elegir `max_features` y evaluar en el test.** Elegimos con validación cruzada cuántas columnas sortea cada corte y evaluamos en el test todos los modelos.

Hay tres ejercicios de código y tres «Para pensar» con la respuesta plegada. En cada ejercicio, los comentarios `# TODO` dicen qué hacer. Si el código repite una celda anterior, indican qué celda copiar y qué cambiar. Si es nuevo, dan una pista con las funciones que hacen falta.

**Solo si trabajas en Google Colab.** La celda siguiente instala la versión de scikit-learn del entorno `tap` del curso. En tu ordenador, con el entorno `tap` activado, no hace nada. Si Colab pide reiniciar la sesión después de instalar, acepta y vuelve a ejecutar el cuaderno desde el principio.

In [ ]:
# Instala las versiones del curso (solo en Google Colab)
import sys
if "google.colab" in sys.modules:
    %pip install -q -r https://raw.githubusercontent.com/juanDmontoro/tap/main/requirements.txt

## 0. Preparación

Usa el entorno `tap` del curso.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# Árboles, bagging y random forest.
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor

# Modelo nulo, regresión lineal y preprocesado de los predictores.
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# División, validación cruzada, búsqueda de hiperparámetros y métricas.
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.metrics import root_mean_squared_error, r2_score

SEMILLA = 42

### Datos y población

Cargamos la EES 2022 desde la web del curso y conservamos la población de los temas 2 y 3: salario positivo y hasta el percentil 99. Trabajamos con los mismos 10 000 trabajadores de `tema_04_arboles`.

In [ ]:
# Datos de la EES 2022 en la web del curso (separador: tabulación).
URL_DATOS = "https://juandmontoro.github.io/tap/data/EES_2022.tab"
df = pd.read_csv(URL_DATOS, sep="\t", low_memory=False)
df.columns = df.columns.str.replace('"', "").str.strip()

# Población de los temas 2 y 3: salario positivo y hasta el percentil 99.
tope = df["RETRINOIN"].quantile(0.99)
salario_admitido = (df["RETRINOIN"] > 0) & (df["RETRINOIN"] <= tope)
limpio = df[salario_admitido]

# 10 000 trabajadores elegidos al azar.
datos = limpio.sample(n=10000, random_state=SEMILLA).reset_index(drop=True)
print("Observaciones:", len(datos))

### Respuesta, predictores y división

La respuesta es el salario bruto anual (`RETRINOIN`, en euros). Usamos los 16 predictores de los temas 2 y 3 y de la entrega E1. Ocho son los de `tema_04_arboles`: estudios (`ESTU`), edad (`ANOS2`), antigüedad (`ANOANTI`), jornada anual (`JAP`), sexo (`SEXO`), tipo de jornada (`TIPOJOR`), tipo de contrato (`TIPOCON`) y supervisión (`RESPONSA`). Los otros ocho son la ocupación (`CNO1`), la actividad económica (`CNACE`), la propiedad pública o privada (`CONTROL`), el mercado de la empresa (`MERCADO`), la forma de regulación de las relaciones laborales (`REGULACION`), la región (`NUTS1`), la nacionalidad (`TIPOPAIS`) y el tamaño de la unidad (`ESTRATO2`). Ningún predictor se calcula con el salario: las componentes del salario, las horas extraordinarias, los identificadores y el factor de elevación quedan fuera, como en el tema 3.

La división es la de `tema_04_arboles`, 80 % de entrenamiento y 20 % de test, con la misma semilla y los mismos trabajadores. El test queda reservado para la evaluación final, en el bloque 2.

In [ ]:
PRED_CATEGORICOS = [
    "SEXO", "ESTU", "ANOS2", "TIPOCON", "TIPOJOR", "CNO1", "CNACE",
    "CONTROL", "MERCADO", "REGULACION", "RESPONSA", "NUTS1", "TIPOPAIS", "ESTRATO2",
]
PRED_NUMERICOS = ["ANOANTI", "JAP"]

# X contiene los 16 predictores: categóricos como categorías y numéricos como decimales (float).
X = datos[PRED_CATEGORICOS + PRED_NUMERICOS].copy()
X[PRED_CATEGORICOS] = X[PRED_CATEGORICOS].astype("category")
X[PRED_NUMERICOS] = X[PRED_NUMERICOS].astype(float)

# y es el salario.
y = datos["RETRINOIN"]

# La división de tema_04_arboles: 80 % entrenamiento y 20 % test.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)
print("Entrenamiento:", len(X_train), "| test:", len(X_test))

### El preprocesador

Cada predictor categórico pasa a una columna indicadora por categoría, y las categorías de menos del 1 % de los trabajadores se agrupan en una. El preprocesador es el habitual con un argumento nuevo, `escalar`. La regresión lineal escala las numéricas (`escalar=True`, el valor por defecto). Los árboles no lo necesitan, porque un corte solo depende del orden de los valores. Con `escalar=False` las numéricas pasan sin cambios y los cortes se leen en años y en horas.

In [ ]:
# Devuelve un preprocesador nuevo, sin ajustar, en cada llamada.
def crear_preprocesador(escalar=True):
    # Numéricas: escaladas, o sin cambios con "passthrough".
    if escalar:
        numericas = StandardScaler()
    else:
        numericas = "passthrough"
    # Categóricas: una columna por categoría (las de menos del 1 % de los trabajadores se agrupan en una).
    return ColumnTransformer([
        ("numericas", numericas, PRED_NUMERICOS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01,
                                      sparse_output=False), PRED_CATEGORICOS),
    ])


# Columnas que salen del preprocesador, ajustado con el entrenamiento.
preprocesador = crear_preprocesador(escalar=False)
preprocesador.fit(X_train)
print("Columnas tras el preprocesador:", len(preprocesador.get_feature_names_out()))

### Validación cruzada y modelo nulo

Comparamos los modelos con el **RMSE de validación cruzada** en el entrenamiento: la media de los 5 RMSE que da `cross_val_score` con las particiones `cv5`, las mismas en todo el cuaderno. El **error típico** de esa media es la desviación típica de los 5 RMSE dividida entre √5. Recuerda que el error típico es orientativo, porque los 5 errores no son independientes.

El **modelo nulo** predice a todos el salario medio de su entrenamiento, sin mirar los predictores. Un modelo útil debe tener un RMSE menor que el suyo.

In [ ]:
# Cinco particiones del entrenamiento, barajadas con la semilla.
cv5 = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)

# Modelo nulo: predice a todos el salario medio de su entrenamiento.
nulo = DummyRegressor(strategy="mean")
puntuaciones_nulo = cross_val_score(nulo, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv5)

# scikit-learn devuelve el RMSE con el signo cambiado: lo pasamos a positivo.
rmse_nulo = -puntuaciones_nulo.mean()
et_nulo = puntuaciones_nulo.std() / np.sqrt(5)
print("Modelo nulo | RMSE CV:", round(rmse_nulo), "euros | error típico:", round(et_nulo))

### El modelo base: la regresión lineal

El **modelo base** es la regresión lineal: el modelo sencillo que un modelo flexible tiene que batir para compensar su complejidad. Compararlos con el mismo protocolo es el patrón del hito H3 de hoy, el modelo base del proyecto.

In [ ]:
# Modelo base: preprocesador con escalado y regresión lineal.
lineal = Pipeline([
    ("preprocesado", crear_preprocesador()),
    ("modelo", LinearRegression()),
])

# RMSE de validación cruzada con las mismas particiones cv5.
puntuaciones_lineal = cross_val_score(lineal, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv5)
print("Regresión lineal | RMSE CV:", round(-puntuaciones_lineal.mean()), "euros | error típico:", round(puntuaciones_lineal.std() / np.sqrt(5)))

**Interpretación.** La regresión lineal baja el RMSE CV del modelo nulo de 18 298 a 11 956 euros. Es la cifra que tienen que mejorar los árboles y los ensambles.

La última celda de la preparación comprueba las versiones de las bibliotecas del tema 4. `xgboost` y `shap` se usarán en las dos próximas semanas. Si la celda da error, falta un paquete del entorno `tap`: avisa al profesor. El resto del cuaderno no depende de ella.

In [ ]:
from importlib.metadata import version

# Versiones instaladas de las tres bibliotecas del tema 4.
print("scikit-learn:", version("scikit-learn"))
print("xgboost:", version("xgboost"))
print("shap:", version("shap"))

# Bloque 1. Del árbol al random forest

## 1.1 El tamaño del árbol, con validación cruzada

`tema_04_arboles` terminaba con una tarea: elegir `min_samples_leaf` con validación cruzada en el entrenamiento. El árbol va en un `Pipeline` con el preprocesador sin escalar. `GridSearchCV` recibe el `Pipeline`, la malla de valores, la métrica y las particiones. Ajusta un árbol por cada valor y cada partición y guarda en `cv_results_` la media y la desviación típica de los RMSE de cada valor ([documentación](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)). En la malla, el nombre del paso del `Pipeline` y el del hiperparámetro van unidos por dos guiones bajos (`modelo__min_samples_leaf`), como en el tema 2. Incluimos `min_samples_leaf=1`, el árbol sin límite.

In [ ]:
# Árbol de regresión con el preprocesador sin escalar.
arbol = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", DecisionTreeRegressor(random_state=SEMILLA)),
])

# Malla: observaciones mínimas en cada hoja. Con 1, el árbol crece sin límite.
malla_arbol = {"modelo__min_samples_leaf": [1, 10, 20, 50, 100, 200]}

busqueda_arbol = GridSearchCV(
    estimator=arbol,
    param_grid=malla_arbol,
    scoring="neg_root_mean_squared_error",
    cv=cv5,
)
busqueda_arbol.fit(X_train, y_train)
print("min_samples_leaf con menor RMSE CV:", busqueda_arbol.best_params_["modelo__min_samples_leaf"])

# Tabla de cv_results_: RMSE en positivo y error típico.
resultados_arbol = pd.DataFrame(busqueda_arbol.cv_results_)
tabla_arbol = resultados_arbol[["param_modelo__min_samples_leaf", "mean_test_score", "std_test_score"]].copy()
tabla_arbol["mean_test_score"] = -tabla_arbol["mean_test_score"]
tabla_arbol["error_tipico"] = tabla_arbol["std_test_score"] / np.sqrt(5)
tabla_arbol.round(0)

**Interpretación.** `mean_test_score` es el RMSE CV en euros, ya en positivo. El menor está en `min_samples_leaf=20` (12 663 euros), y el árbol sin límite (`min_samples_leaf=1`) es el peor, con unos 16 100 euros: es el sobreajuste de `tema_04_arboles`, ahora medido con validación cruzada.

> **Cifras en otro ordenador.** Los modelos con árboles sin límite (este árbol, el bagging y el random forest) pueden dar en tu ordenador un RMSE que se separa unas decenas de euros del que cita el texto, aunque la semilla sea la misma, y la predicción de un árbol suelto puede cambiar más. En sus hojas, de uno o dos trabajadores, varios cortes empatan, y cada ordenador deshace el empate según diferencias mínimas de redondeo. Por eso el texto da redondeadas las cifras de esos modelos. Las comparaciones entre modelos se mantienen.

La **regla del error típico** (tema 2) elige el árbol más sencillo cuyo RMSE CV no supera el mínimo más su error típico. Con un `min_samples_leaf` mayor, cada hoja reúne más trabajadores y el árbol tiene menos hojas, así que es más sencillo.

In [ ]:
# RMSE CV y error típico del árbol con menor RMSE: la fila best_index_ de la tabla.
rmse_arbol = tabla_arbol.loc[busqueda_arbol.best_index_, "mean_test_score"]
et_arbol = tabla_arbol.loc[busqueda_arbol.best_index_, "error_tipico"]

# Límite de la regla: el RMSE mínimo más su error típico.
limite_arbol = rmse_arbol + et_arbol

# Árboles con RMSE dentro del límite. El más sencillo es el de mayor min_samples_leaf.
dentro_arbol = tabla_arbol[tabla_arbol["mean_test_score"] <= limite_arbol]
print("Límite:", round(limite_arbol), "euros")
print("Mayor min_samples_leaf dentro del límite:", dentro_arbol["param_modelo__min_samples_leaf"].max())

**Interpretación.** El límite es 12 868 euros, y el árbol más sencillo dentro de él es el de `min_samples_leaf=50` (12 682 euros), 19 euros por encima del mínimo. Los dos criterios eligen árboles distintos con un RMSE CV casi igual. Como referencia usamos el del mínimo, que `GridSearchCV` guarda en `best_estimator_`. Los dos quedan por encima de la regresión lineal (11 956 euros).

## 1.2 Bagging

El **bagging** ajusta B árboles, cada uno con un remuestreo bootstrap del entrenamiento, y predice la media de sus B predicciones. Usamos 100 árboles sin límite, como el de `min_samples_leaf=1` de la tabla anterior, con el mismo preprocesador.

> **Python que usamos aquí**
> - `BaggingRegressor(estimator=modelo, n_estimators=B, random_state=semilla)` crea un bagging de B copias de `modelo`, cada una ajustada con su propio remuestreo ([documentación](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingRegressor.html)).

In [ ]:
# Bagging de 100 árboles sin límite, con el preprocesador sin escalar.
bagging = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", BaggingRegressor(estimator=DecisionTreeRegressor(), n_estimators=100, random_state=SEMILLA)),
])

# RMSE de validación cruzada con las particiones cv5.
puntuaciones_bagging = cross_val_score(bagging, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv5)
print("Bagging | RMSE CV:", round(-puntuaciones_bagging.mean()), "euros | error típico:", round(puntuaciones_bagging.std() / np.sqrt(5)))

**Interpretación.** El bagging baja el RMSE CV del árbol sin límite de unos 16 100 a unos 11 500 euros. Mejora también al árbol de `min_samples_leaf=20` (12 663 euros) y a la regresión lineal (11 956 euros), en los dos casos por más de dos errores típicos.

## 1.3 Dentro del bagging

Ajustamos el bagging una vez con todo el entrenamiento para ver sus remuestreos y sus árboles.

> **Python que usamos aquí**
> - `np.unique(array)` devuelve los valores distintos de `array`, sin repeticiones.

In [ ]:
# Un ajuste con todo el entrenamiento.
bagging.fit(X_train, y_train)

# bagging["modelo"]: el paso "modelo" del Pipeline, es decir, el bagging ajustado.
bagging_ajustado = bagging["modelo"]

# estimators_samples_[0]: posiciones de los trabajadores del remuestreo del primer árbol, con repeticiones.
indices_arbol_1 = bagging_ajustado.estimators_samples_[0]

# Fracción de trabajadores distintos en ese remuestreo.
fraccion_distintos = len(np.unique(indices_arbol_1)) / len(X_train)
print("Trabajadores distintos en el primer remuestreo:", round(fraccion_distintos, 3))

**Interpretación.** El primer remuestreo contiene 8 000 posiciones, pero solo el 63,2 % de los trabajadores del entrenamiento: unos salen una vez, otros varias, y el resto queda **fuera de la bolsa** de ese árbol. Es la proporción que se espera en un remuestreo bootstrap de muchas observaciones, 1 − e⁻¹ ≈ 0,632.

Ahora miramos las predicciones para un trabajador del test, el primero con licenciatura o doctorado (`ESTU = 7`). Solo miramos sus predicciones: el test se usa para evaluar los modelos al final del bloque 2.

In [ ]:
# Primer trabajador del test con ESTU = 7 (doble corchete: se conserva como tabla).
licenciados_test = X_test[X_test["ESTU"] == 7]
trabajador = licenciados_test.iloc[[0]]

# Los árboles del bagging reciben las columnas del preprocesador: transformamos la fila.
fila = bagging["preprocesado"].transform(trabajador)

# estimators_[k]: árbol k del bagging (el primero es el 0). Predicciones de tres árboles y del bagging.
print("Árbol 1:", bagging_ajustado.estimators_[0].predict(fila).round())
print("Árbol 2:", bagging_ajustado.estimators_[1].predict(fila).round())
print("Árbol 3:", bagging_ajustado.estimators_[2].predict(fila).round())
print("Árbol 4:", bagging_ajustado.estimators_[3].predict(fila).round())
print("Árbol 5:", bagging_ajustado.estimators_[4].predict(fila).round())
print("Bagging (media de los 100 árboles):", bagging.predict(trabajador).round())

**Interpretación.** Entre la mayor y la menor de las tres predicciones para el mismo trabajador hay miles de euros. Cada árbol se ajustó con otro remuestreo y termina en hojas de muy pocos trabajadores, así que su predicción depende de los trabajadores que le tocaron. El bagging predice la media de los 100 árboles.

### Para pensar 1

Si pasamos de 100 a 300 árboles, ¿puede el bagging sobreajustar?

<details><summary>Respuesta</summary>

No. Cada árbol conserva su profundidad, y añadir árboles solo acerca la media a su límite. La varianza de la media de B árboles es $\bar\rho\,\sigma^2 + (1-\bar\rho)\,\sigma^2/B$, donde $\sigma^2$ es la varianza de la predicción de un árbol y $\bar\rho$, la correlación media entre las predicciones de dos árboles. Al añadir árboles solo baja el segundo término, y la varianza se estabiliza en $\bar\rho\,\sigma^2$. El RMSE de validación cruzada baja y se estabiliza, aunque puede fluctuar un poco. Puedes comprobarlo con `n_estimators=300`.

</details>

## 1.4 Random forest

El **random forest** es un bagging de árboles en el que cada corte elige entre una fracción de las columnas, sorteada de nuevo en cada corte: es el hiperparámetro `max_features`. Con el preprocesador, las columnas son las 88 de la preparación. Con `max_features=1.0` cada corte ve todas las columnas, como en el bagging.

> **Python que usamos aquí**
> - `RandomForestRegressor(n_estimators=B, max_features=fracción, oob_score=métrica, random_state=semilla)` crea un random forest de B árboles ([documentación](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)). Con `oob_score=root_mean_squared_error` calcula además el RMSE fuera de la bolsa.

In [ ]:
# Random forest: 100 árboles, y cada corte elige entre el 33 % de las columnas.
bosque = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", RandomForestRegressor(n_estimators=100, max_features=0.33, random_state=SEMILLA)),
])

# RMSE de validación cruzada con las particiones cv5.
puntuaciones_bosque = cross_val_score(bosque, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv5)
print("Random forest 0,33 | RMSE CV:", round(-puntuaciones_bosque.mean()), "euros | error típico:", round(puntuaciones_bosque.std() / np.sqrt(5)))

El **error fuera de la bolsa** (OOB) predice cada trabajador del entrenamiento con los árboles que no lo usaron y compara esas predicciones con su salario. Sale de un único ajuste del bosque con todo el entrenamiento.

In [ ]:
# El mismo bosque, ajustado una vez con todo el entrenamiento, con el RMSE fuera de la bolsa.
bosque_oob = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", RandomForestRegressor(n_estimators=100, max_features=0.33,
                                     oob_score=root_mean_squared_error, random_state=SEMILLA)),
])
bosque_oob.fit(X_train, y_train)

# bosque_oob["modelo"]: el paso "modelo" del Pipeline, es decir, el random forest ajustado.
bosque_ajustado = bosque_oob["modelo"]

# oob_score_: su RMSE fuera de la bolsa, en euros.
print("Random forest 0,33 | RMSE OOB:", round(bosque_ajustado.oob_score_), "euros")

**Interpretación.** El RMSE OOB y el de validación cruzada casi coinciden, en unos 11 200 euros. El OOB sale de un ajuste con los 8 000 trabajadores del entrenamiento, y la validación cruzada, de cinco ajustes con 6 400. El random forest mejora al bagging (unos 11 500 euros) en algo más que el error típico de cualquiera de los dos, que ronda los 200 euros.

### Ejercicio 1

1. Ajusta un random forest de 100 árboles con `max_features=1.0` y calcula su RMSE de validación cruzada y su error típico con `cv5`.
2. Compáralo con el bagging (1.2) y con el random forest de 0,33. ¿Qué relación hay entre el random forest con `max_features=1.0` y el bagging? ¿La diferencia con el de 0,33 supera el error típico?

In [ ]:
# TODO: copia la celda del random forest de 0,33 del apartado 1.4 (la de cross_val_score)
#   y cambia max_features a 1.0. Llama al Pipeline bosque_todos
#   y a sus puntuaciones, puntuaciones_todos.

print("Random forest 1,0 | RMSE CV:", round(-puntuaciones_todos.mean()), "euros | error típico:", round(puntuaciones_todos.std() / np.sqrt(5)))

Paso 2. Compara el random forest con `max_features=1.0` con el bagging y con el random forest de 0,33.

*Escribe aquí tu respuesta.*

### Para pensar 2

¿Por qué sortear las columnas de cada corte hace que los árboles se parezcan menos?

<details><summary>Respuesta</summary>

En el bagging, cada corte elige entre las 88 columnas, y todos los árboles empiezan cortando por una de dos variables: la licenciatura o doctorado (`ESTU = 7`) o el tipo de jornada (`TIPOJOR`), las que más reducen la RSS. Con `max_features=0.33`, scikit-learn sortea 29 columnas en cada corte (redondea 0,33 × 88 hacia abajo), y esas dos variables no siempre están entre ellas. Los primeros cortes del random forest se reparten entre diez columnas. Los árboles cortan por variables distintas, sus errores se parecen menos (baja $\bar\rho$) y baja el término $\bar\rho\,\sigma^2$, que el número de árboles no reduce. Cada árbol, a cambio, puede predecir algo peor, y por eso la fracción se elige por validación cruzada (bloque 2).

</details>

# Bloque 2. Elegir `max_features` y evaluar en el test

## 2.1 La tabla de comparación

Reunimos los RMSE de validación cruzada de los modelos de los apartados anteriores con las particiones `cv5`.

In [ ]:
# Resumen de los modelos ajustados hasta ahora, todos con las particiones cv5.
resumen = pd.DataFrame({
    "modelo": ["nulo", "regresión lineal", "árbol elegido en 1.1", "bagging", "random forest 0,33"],
    "RMSE CV (€)": [rmse_nulo, -puntuaciones_lineal.mean(), rmse_arbol,
                    -puntuaciones_bagging.mean(), -puntuaciones_bosque.mean()],
    "error típico (€)": [et_nulo, puntuaciones_lineal.std() / np.sqrt(5), et_arbol,
                         puntuaciones_bagging.std() / np.sqrt(5), puntuaciones_bosque.std() / np.sqrt(5)],
})
resumen.round(0)

**Interpretación.** El random forest de 0,33 tiene el menor RMSE CV (unos 11 200 euros), unos 750 euros por debajo de la regresión lineal (11 956 euros), más de tres veces el error típico de cualquiera de los dos. El bagging mejora a la regresión lineal en unos 500 euros. El árbol elegido en 1.1 (12 663 euros) queda 707 euros por encima de ella.

## 2.2 ¿Cuántas columnas por corte?

Hasta ahora hemos probado dos valores de `max_features`: 0,33 y 1,0, que es el bagging. Es el hiperparámetro principal del random forest, y lo elegimos con validación cruzada, como el tamaño del árbol en 1.1.

La regla del error típico de 1.1 elige el modelo más sencillo, y para eso hay que saber qué valor del hiperparámetro da un modelo más sencillo. Con `min_samples_leaf`, un valor mayor da árboles con menos hojas. Con `max_features`, cualquier fracción da 100 árboles que crecen sin límite y tienen el mismo tamaño, y no hay un orden de sencillez aceptado entre sus valores. Por eso, elegimos `max_features` por el mínimo de la validación cruzada. El límite de la regla sirve para ver qué valores quedan a menos de un error típico del mínimo, los que la validación cruzada no distingue de él.

### Ejercicio 2. Búsqueda de `max_features`

1. Busca `max_features` con `GridSearchCV` en la malla 0,1, 0,2, 0,33, 0,5 y 1,0, con un random forest de 100 árboles en un `Pipeline` con el preprocesador sin escalar, y muestra la tabla de `cv_results_` con el error típico.
2. Calcula el límite de la regla del error típico (el RMSE mínimo más su error típico) y los valores de `max_features` cuyo RMSE CV no lo supera.
3. ¿Qué `max_features` tiene el menor RMSE CV? ¿Cuántos valores quedan dentro del límite y qué indica eso? Compara el mínimo con el random forest de 0,33 y con la regresión lineal de la tabla 2.1.

In [ ]:
# TODO: copia la celda de la búsqueda del árbol (apartado 1.1, la de GridSearchCV) y cambia:
#   - el modelo del Pipeline: RandomForestRegressor(n_estimators=100, random_state=SEMILLA),
#     sin max_features, porque lo fija la búsqueda.
#   - la malla: {"modelo__max_features": [0.1, 0.2, 0.33, 0.5, 1.0]}.
#   - los nombres: bosque_malla, malla_columnas, busqueda_columnas, resultados_columnas
#     y tabla_columnas, con la columna "param_modelo__max_features".
#   - el print de best_params_: su texto y el nombre entre corchetes, "modelo__max_features".
#   - elimina la última línea (tabla_arbol.round(0)). Usa la línea de abajo que muestra la tabla.

# Redondeamos solo los RMSE: max_features tiene decimales.
display(tabla_columnas.round({"mean_test_score": 0, "std_test_score": 0, "error_tipico": 0}))

In [ ]:
# TODO: copia la celda de la regla del error típico (apartado 1.1) y cámbiala para la búsqueda
#   de max_features: busqueda_columnas y tabla_columnas en lugar de busqueda_arbol y tabla_arbol.
#   Nombres nuevos: rmse_columnas, et_columnas, limite_columnas y dentro_columnas.
# TODO: cambia la última línea. En lugar del mayor min_samples_leaf, imprime los valores
#   de max_features que quedan dentro del límite.
#   Pista: dentro_columnas["param_modelo__max_features"].tolist() los devuelve como lista.

Paso 3. ¿Qué `max_features` tiene el menor RMSE CV? ¿Cuántos valores quedan dentro del límite y qué indica eso? Compara el mínimo con el random forest de 0,33 y con la regresión lineal.

*Escribe aquí tu respuesta.*

## 2.3 Evaluación en el test

Hemos elegido el modelo con validación cruzada, sin mirar el test: el random forest con el `max_features` del mínimo del ejercicio 2. Ahora calculamos, una sola vez, el RMSE y el R² en el test de todos los modelos, para ver si el orden de la validación cruzada se mantiene con 2 000 trabajadores que ningún modelo ha visto. El test informa de los resultados y no cambia la elección.

El **R²** es 1 menos el cociente entre la suma de los errores al cuadrado del modelo y la suma de las desviaciones al cuadrado del salario respecto a su media en el test. Vale 0 si el modelo predice como la media y 1 si acierta a todos.

> **Python que usamos aquí**
> - `r2_score(observado, predicho)` devuelve el R² de las predicciones ([documentación](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)).

Empezamos por la regresión lineal: la ajustamos con todo el entrenamiento y la evaluamos en el test.

In [ ]:
# Ajuste de la regresión lineal con todo el entrenamiento.
lineal.fit(X_train, y_train)

# Predicciones en el test.
pred_lineal = lineal.predict(X_test)

# RMSE y R² en el test.
rmse_test_lineal = root_mean_squared_error(y_test, pred_lineal)
r2_test_lineal = r2_score(y_test, pred_lineal)
print("Regresión lineal | RMSE test:", round(rmse_test_lineal), "euros | R² test:", round(r2_test_lineal, 3))

### Ejercicio 3. Todos los modelos en el test

1. Ajusta el modelo nulo con todo el entrenamiento, que es el único que falta, y reúne en un diccionario los seis modelos ajustados, en este orden: nulo, regresión lineal, árbol elegido (`busqueda_arbol.best_estimator_`), bagging (`bagging`, ajustado en 1.3), random forest de 0,33 (`bosque_oob`, ajustado en 1.4) y random forest elegido en el ejercicio 2 (`busqueda_columnas.best_estimator_`).
2. Para cada modelo, calcula el RMSE y el R² en el test como en el caso de la regresión lineal y reúnelos en una tabla. Las últimas líneas de la celda añaden el RMSE CV y marcan el modelo elegido por validación cruzada, el de menor RMSE CV.
3. ¿Se mantiene en el test el orden de la validación cruzada? ¿Cuánto mejora el modelo elegido a la regresión lineal en RMSE y en R²?

In [ ]:
# TODO 1: ajusta el modelo nulo con todo el entrenamiento y crea el diccionario modelos,
#   {nombre: modelo}, con los seis modelos del enunciado, en ese orden.
# TODO 2: para cada modelo, predicciones en el test, RMSE y R², con las líneas del caso de la
#   regresión lineal. Reúnelos en la tabla tabla_test, con las columnas "modelo",
#   "RMSE test (€)" y "R² test".
#   Pista: un bucle for nombre, modelo in modelos.items(): que añade una fila a la lista filas
#   con filas.append({"modelo": nombre, ...}). Después, tabla_test = pd.DataFrame(filas).

# RMSE CV de los seis modelos, en el orden del diccionario.
tabla_test["RMSE CV (€)"] = [rmse_nulo, -puntuaciones_lineal.mean(), rmse_arbol,
                             -puntuaciones_bagging.mean(), -puntuaciones_bosque.mean(), rmse_columnas]

# Elegido por validación cruzada: el de menor RMSE CV.
tabla_test["elegido por CV"] = tabla_test["RMSE CV (€)"] == tabla_test["RMSE CV (€)"].min()
display(tabla_test.round({"RMSE test (€)": 0, "R² test": 3, "RMSE CV (€)": 0}))

Paso 3. ¿Se mantiene en el test el orden de la validación cruzada? ¿Cuánto mejora el modelo elegido a la regresión lineal en RMSE y en R²?

*Escribe aquí tu respuesta.*

### Para pensar 3

En el hito H3 preparáis el modelo base de vuestro proyecto. ¿Qué modelo usaríais de base y cómo compararíais con él un random forest?

<details><summary>Respuesta</summary>

El modelo base es un modelo sencillo con vuestra respuesta y vuestros predictores: una regresión lineal si la respuesta es numérica o un logit si es binaria (tema 3). Por debajo queda el modelo nulo. El random forest se compara con el protocolo de este cuaderno: la misma división entre entrenamiento y test, las mismas particiones de validación cruzada en el entrenamiento, la misma métrica con su error típico y el preprocesado dentro de un `Pipeline`. Se elige con validación cruzada, y el test se usa una sola vez, al final, para informar de los resultados.

</details>

# Qué viene después

- El martes 3-11 vemos el **boosting**, que también combina árboles: los ajusta uno detrás de otro, y cada árbol nuevo corrige los errores de los anteriores.
- `xgboost` y `shap`, cuyas versiones comprobaste en la preparación, son las bibliotecas de las dos próximas semanas.